# Kütüphaneler

In [14]:
import pandas as pd
import numpy as np
import pandas.io.formats.excel as pife
import warnings
import pickle
import os
import locale
from termcolor import colored
from pandas.errors import SettingWithCopyWarning
from datetime import datetime

# Ayarlar

In [15]:
os.system('color')
locale.setlocale(locale.LC_ALL, 'tr_TR')
pd.options.display.float_format = '{:,.2f}'.format
pife.ExcelFormatter.header_style = None

warnings.filterwarnings('ignore', category = UserWarning)
warnings.filterwarnings('ignore', category = pd.errors.ParserWarning)
warnings.filterwarnings('ignore', category = SettingWithCopyWarning)

base_path = 'C:\\Users\\105617\\Desktop\\Workspace\\Panel\\'

pickle_folder_path = base_path + 'pickle' + '\\'
data_folder_path = base_path + 'data' + '\\'
output_folder_path = base_path + 'output' + '\\'

# Yardımcı Fonksiyonlar

## Tablo Manipülasyonu

In [16]:
def add_h_space(df_original, space=1):
    df_final = df_original.copy()
    width = df_final.shape[1]
    for _ in range(space):
        df_final.loc[len(df_final)] = [np.nan] * width
    return df_final

def add_v_space(df_original, space=1):
    df_final = df_original.copy()
    df_null = pd.DataFrame(columns=[np.nan] * space)
    df_final = pd.concat([df_final, df_null], axis=1)
    return df_final
    

def h_stack(df_list, space=3):
    df_first = df_list[0].copy().reset_index(drop=True)
    df_list_new = [df_first]
    df_null = pd.DataFrame({np.nan: [np.nan]})
    for df in df_list[1:]:
        df_temp = df.copy()
        for _ in range(space):
            df_list_new.append(df_null)
        df_list_new.append(df_temp.reset_index(drop=True))
    df_final = pd.concat(df_list_new, axis=1).reset_index(drop=True)
    return df_final


def v_stack(df_list, space=3):
    max_width = max([x.shape[1] for x in df_list])

    df_first = df_list[0].copy().reset_index(drop=True)
    if df_first.shape[1] < max_width:
        df_first = add_v_space(df_first, max_width - df_first.shape[1])
    df_list_new = [df_first.copy()]
    
    df_null = pd.DataFrame(columns=df_first.columns)
    df_null = add_h_space(df_null, space)

    for df in df_list[1:]:
        df_temp = df.copy()
        if df_temp.shape[1] < max_width:
            df_temp = add_v_space(df, max_width - df_temp.shape[1])

        df_new = pd.DataFrame([df_temp.columns], columns=df_first.columns)
        df_temp.columns = df_first.columns

        df_list_new.append(df_null)
        df_list_new.append(df_new)
        df_list_new.append(df_temp.reset_index(drop=True))

    df_final = pd.concat(df_list_new).reset_index(drop=True)
    return df_final


def insert_header(df_original, column):
    df_final = df_original.copy()
    header = [column] if type(column) is str else column
    df_final = v_stack([pd.DataFrame(columns=header), df_final], 0)
    return df_final

def insert_corner(df_original):
    df_final = df_original.copy()
    df_final = h_stack([pd.DataFrame(), v_stack([pd.DataFrame(), df_final], 0)], 1)
    return df_final

def insert_row(df_original, index=0, row=None, count=1):
    df_final = df_original.copy().reset_index(drop=True)
    
    if row is None:
        row = np.nan
    if type(row) is list and len(row) < df_final.shape[1]:
        row += [np.nan] * (df_final.shape[1] - len(row))
    if index < 0:
        index += df_final.shape[0]

    new_indices = [x for x in range(index)] + [x + count for x in range(index, len(df_final))]
    df_final.index = new_indices

    for i in range(count):
        df_final.loc[index + i] = row

    df_final = df_final.reset_index().sort_values('index').drop('index', axis=1).reset_index(drop=True)
    return df_final

## Verim Tabloları

In [17]:
def fix_verim_table(df, columns=None):
    start = 0
    if columns is None:
        start = 3
        columns = df.iloc[2, 1:]
    df = df.iloc[start:, 1:]
    df.columns = columns
    df.columns.name = None
    df = df.reset_index(drop=True)
    return df

def quick_export_verim_csv(data, output_file_name, sep=';', header=False, index=False, open_file=False):
    cl = [str(x) for x in range(0, 16)]
    df = pd.DataFrame()
    df['10'] = data
    for c in ['0', '15']:
        df[c] = 'x'
    for c in cl:
        if c not in ['0', '10', '15']:
            df[c] = np.nan
    df = df[cl]

    output_file_path = output_folder_path + output_file_name + '.csv'
    df.to_csv(output_file_path, sep=sep, header=header, index=index)

    if open_file:
        os.startfile(output_file_path)
    

## Hızlı Çıktı

In [18]:
def quick_export(df_export, output_file_name, sheet_name=None, open_file=True, hide_gridlines=True):
    output_file_path = output_folder_path + output_file_name + '.xlsx'
    writer = pd.ExcelWriter(output_file_path, engine = 'xlsxwriter')
    if type(df_export) is list:
        if sheet_name is not None:
            sheet_list = sheet_name
        else:
            sheet_list = [str(x + 1) for x in range(len(df_export))]
        for df, sheet in zip(df_export, sheet_list):
            df.to_excel(writer, sheet_name=sheet, index=False)
            if hide_gridlines:
                worksheet = writer.sheets[sheet]
                worksheet.hide_gridlines(2)
    else:
        if sheet_name is None:
            sheet = output_file_name[:20]
        else:
            sheet = sheet_name[:20]
        df_export.to_excel(writer, sheet_name=sheet, index=False)
        if hide_gridlines:
            worksheet = writer.sheets[sheet]
            worksheet.hide_gridlines(2)
    writer.close()
    if open_file:
        os.startfile(output_file_path)


def quick_export_csv(df, file_name, open_file=False):
    df.to_csv(output_folder_path + file_name  + '.csv', sep=';', encoding='ANSI', index=False)

## Now

In [19]:
def now():
    # return datetime.strftime(datetime.now(), '%Y-%m-%d_%H-%M-%S')
    return datetime.strftime(datetime.now(), '%m%d%H%M')

## Pickle

In [20]:
def save_pickle(file, name, sub=''):
    os.makedirs(pickle_folder_path + sub, exist_ok = True)
    with open(pickle_folder_path + sub + name, 'wb') as handle:
        pickle.dump(file, handle, protocol=pickle.HIGHEST_PROTOCOL)

def load_pickle(name, sub=''):
    with open(pickle_folder_path + sub + name, 'rb') as handle:
        return pickle.load(handle)

# Listeler

In [21]:
segment_list, segment_title_list, segment_title_dict, bolum_list, filtered_bolum_list, sektor_list, filtered_sektor_list, sektor_title_dict, sektor_title_short_dict, sube_list, bolge_list = load_pickle('list_list')

In [22]:
risk_category_list = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek']

new_filtered_bolum_list = [
    'Kurumsal Krediler Tahsis Bölümü',
    'Ticari Krediler Tahsis Bölümü',
    'PKTB - Bölge Yetkili',
    'PKTB - Şube Yetkili',
]

# Validasyon

## Max MS

In [23]:
def find_max_column(df, columns):
    values = [df[c] for c in columns if not np.isnan(df[c])]
    if len(values): return max(values)
    else: return np.nan
    
def find_first_occurrence(df, columns, threshold):  
    return list(df[columns].values).index(threshold) + 1

def get_df_pra_max(df_pra, date_list, column='Müşteri Sınıfı'):
    c = column
    sorted_date_list = date_list
    mn = 'Müşteri No'
    a = 'Adı'

    df = df_pra.copy()
    si = sorted_date_list.index(start_term) + 1
    ei = sorted_date_list.index(end_term) + 1
    dfl = df_wide[si:ei]
    dl = sorted_date_list[si:ei]

    for dft, d in zip(dfl, dl):
        df = pd.merge(df, dft[['Müşteri No', 'Müşteri Sınıfı ' + d]], on=mn, how='left')

    cs = [c + ' ' + d for d in dl]
    df[c + ' Max'] = df.apply(find_max_column, columns=cs, axis=1)
    df['2. Sınıfa Geçiş Ayı'] = np.nan
    df['3. Sınıfa Geçiş Ayı'] = np.nan

    df.loc[df[c + ' Max'] == 2, '2. Sınıfa Geçiş Ayı'] = df.loc[df[c + ' Max'] == 2].apply(find_first_occurrence, columns=cs, threshold=2, axis=1)
    df.loc[df[c + ' Max'] == 3, '3. Sınıfa Geçiş Ayı'] = df.loc[df[c + ' Max'] == 3].apply(find_first_occurrence, columns=cs, threshold=3, axis=1)

    return df.copy()

# PRA

In [24]:
pra_v, start_term, end_term, last_date = 'v12.1', '2022.12', '2023.12', '2023.12'
df_wide = load_pickle('df_ms_all_2212_2312')
sorted_date_list = load_pickle('sorted_date_list_2212_2312')
df_old = load_pickle('df_standard_2212')

In [25]:
def fix_bolum(df):
    if df['İlgili Bölüm'] == 'Perakende Krediler Tahsis Bölümü':
        if df['Yetki Kodu'] == 1:
            return 'PKTB - Şube Yetkili'
        else:
            return 'PKTB - Bölge Yetkili'
    else:
        return df['İlgili Bölüm']

df_old = get_df_pra_max(df_old, sorted_date_list)
df_old['İlgili Bölüm'] = df_old.apply(fix_bolum, axis=1)

# Özet

In [26]:
def create_pra_summary(df_pra):
    dfp = df_pra.copy()
    rk_list = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek']
    rk = 'Risk Kategorisi'
    factor = 1_000_000

    t_list = ['Genel Toplam']
    df = pd.DataFrame()
    df[rk] = rk_list
    df = add_v_space(df)
    t_list.append(np.nan)

    c = 'Müşteri No'
    n = 'Adet'
    p = 'Adet Payı'
    dft = dfp[[rk, c]].groupby(rk).count().reset_index().rename(columns={c: n})
    df = pd.merge(df, dft, on=rk, how='left')
    t = df[n].sum()
    t_list.append(t)
    df[p] = df[n] / t
    t_list.append(1)

    df = add_v_space(df)
    t_list.append(np.nan)

    c = 'Nakdi Reeskontlu Toplam'
    n = 'Nakdi Risk'
    p = 'Nakdi Risk Payı'
    dft = dfp[[rk, c]].groupby(rk).sum().reset_index().rename(columns={c: n})
    dft[n] /= factor
    df = pd.merge(df, dft, on=rk, how='left')
    t = df[n].sum()
    t_list.append(t)
    df[p] = df[n] / t
    t_list.append(1)

    df = add_v_space(df)
    t_list.append(np.nan)

    c = 'Entegre Derece Skor'
    n = 'Ortalama Entegre Derece / Skor'
    dft = dfp[[rk, c]].groupby(rk).mean().reset_index().rename(columns={c: n})
    df = pd.merge(df, dft, on=rk, how='left')
    t = dfp[c].mean()
    t_list.append(t)

    c = 'Yapay Zeka Risk Sınıfı'
    n = 'Ortalama Yapay Zeka Risk Sınıfı'
    dft = dfp[[rk, c]].groupby(rk).mean().reset_index().rename(columns={c: n})
    df = pd.merge(df, dft, on=rk, how='left')
    t = dfp[c].mean()
    t_list.append(t)

    df = add_h_space(df)
    df.loc[len(df)] = t_list
    df.loc[len(df)] = [f'{last_date} KR202, Reeskontlu, mio TL'] + [np.nan] * (len(df.columns) - 1)

    return insert_corner(df)

# Geçiş

In [27]:
def create_pra_pass_alt(df_old_max, rows, row_list, row_new=None, last_date=last_date, drop_na=True, replace_na=True, find_risk=False):
    r = rows
    r_list = row_list
    r_dict = row_new
    mn = 'Müşteri No'
    nrt = 'Nakdi Reeskontlu Toplam'
    msm = 'Müşteri Sınıfı Max'
    msl = 'Müşteri Sınıfı ' + last_date
    factor = 1_000_000

    if find_risk:
        x = nrt
        s = 'Nakdi Risk'
    else:
        x = mn
        s = 'Adet'

    dfp = df_old_max.copy()
    if find_risk:
        dfp[nrt] /= factor

    t_list = ['Genel Toplam']
    df = pd.DataFrame()
    df[r] = r_list

    c = x
    n = s
    p = '%'
    dft = dfp[[r, c]].copy()
    if find_risk:
        dft = dft.groupby(r).sum().reset_index().rename(columns={c: n})
    else:
        dft = dft.groupby(r).count().reset_index().rename(columns={c: n})
    df = pd.merge(df, dft, on=r, how='left')
    t = df[n].sum()
    t_list.append(t)
    df[p] = df[n] / t
    t_list.append(1)

    df = add_v_space(df)
    t_list.append(np.nan)

    for ss in [2, 3]:
        c = x
        n = f'{ss}. Sınıfa Geçen {s}'
        ga = f'{ss}. Sınıfa Geçiş Ayı'
        dfa = dfp.loc[dfp[msm] == ss, [r, c, ga]].copy()
        if find_risk:
            dft = dfa[[r, c]].groupby(r).sum().reset_index().rename(columns={c: n})
        else:
            dft = dfa[[r, c]].groupby(r).count().reset_index().rename(columns={c: n})
        df = pd.merge(df, dft, on=r, how='left')
        t = df[n].sum()
        t_list.append(t)

        p = f'{ss}. Sınıfa Geçiş %'
        df[p] = df[n] / df[s]
        t = df[n].sum() / df[s].sum()
        t_list.append(t)

        c = f'{ss}. Sınıfa Geçiş Ayı'
        n = f'Ortalama {ss}. Sınıfa Geçiş Ayı'
        t = dfa[ga].mean()
        t_list.append(t)
        dft = dfa[[r, ga]].groupby(r).mean().reset_index().rename(columns={c: n})
        df = pd.merge(df, dft, on=r, how='left')

        df = add_v_space(df)
        t_list.append(np.nan)

    c = x
    n = 'Riski Kapanan'
    dfa = dfp.loc[dfp[msl].isnull(), [r, c]].copy()
    if find_risk:
        dft = dfa.groupby(r).sum().reset_index().rename(columns={c: n})
    else:
        dft = dfa.groupby(r).count().reset_index().rename(columns={c: n})
    df = pd.merge(df, dft, on=r, how='left')
    t = df[n].sum()
    t_list.append(t)

    p = 'Riski Kapanma %'
    df[p] = df[n] / df[s]
    t = df[n].sum() / df[s].sum()
    t_list.append(t)

    if drop_na:
        df = df.dropna(subset=list(df.columns)[2:], how='all').reset_index(drop=True)

    if r_dict is not None:
        if type(r_dict) is list:
            r_dict = dict(zip(r_list, r_dict))
        df[r] = df[r].apply(lambda x: r_dict.get(x, x))

    df = add_h_space(df)
    df.loc[len(df)] = t_list

    if replace_na:
        df = df.replace(np.nan, '')
        # df.columns = [x if pd.notnull(x) else '' for x in df.columns]

    # header = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
    # df = insert_header(df, header)
    # footer_end = 'Reeskontlu, mio TL' if find_risk else 'Adet'
    # footer = f'{start_term}-{end_term} KR202, {footer_end}'
    # df.loc[len(df)] = [footer] + [np.nan] * (len(df.columns) - 1)
    # df.loc[len(df)] = ['Riski kapanan firmalar dahildir'] + [np.nan] * (len(df.columns) - 1)

    return df.copy()


In [28]:
def categorize_pra(df, x=None, threshold_list=None, category_list=None):
    default_x = 'Toplam Puan'
    default_threshold_list = [0, 2, 5, 10, 15, np.inf]
    default_category_list = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek']

    if x is None:
        x = default_x
    if threshold_list is None:
        threshold_list = default_threshold_list
    if category_list is None:
        category_list = default_category_list

    if df[x] == np.inf: return category_list[-1]
    for t, c in zip(threshold_list, category_list):
        if df[x] <= t: return c
    return category_list[-1]
    
def create_pra_risk_categories(dfp, x='Toplam Puan', threshold_list=None, category_list=None, r='Risk Kategorisi', xx=None, factor=1_000_000):
    m = 'Müşteri No'
    c = 'Nakdi Reeskontlu Toplam'
    nn = 'Adet'
    s = 'Kümülatif Risk'
    ss = 'Kümülatif Adet'
    p = '%'
    pp = '% '
    if xx is None: xx=x

    df = dfp[[x, c]].copy()
    df = df.groupby(x).sum().reset_index()
    df[c] /= factor
    df[s] = df[c].cumsum()
    t = df[s].values[-1]
    df[p] = df[s] / t
    df[r] = df.apply(categorize_pra, x=xx, threshold_list=threshold_list, category_list=category_list, axis=1)
    
    dft = dfp[[m, x]].copy()
    dft = dft.groupby(x).count().reset_index().rename(columns={m: nn})
    dft[ss] = dft[nn].cumsum()
    tt = dft[ss].values[-1]
    dft[pp] = dft[ss] / tt

    df[' '] = np.nan
    df = pd.merge(df, dft, on=x)

    return df


def return_excluded_info(name, df, minus=True, factor=1_000_000):
    sign = -1 if minus else 1
    return [name, sign * df[mn].count(), sign * df[nr].sum() / factor, sign * df[gnr].sum() / factor, sign * df[tr].sum() / factor]

def return_percentages(c):
    percentages = list(dfk.loc[dfk['Küme'] == c].values[0][1:] / dfk.loc[dfk['Küme'] == c0].values[0][1:])
    return [' '] + percentages

# Puanlama

In [35]:
def fg_score(df):
    return int(df['FG Kriter Sayısı'])

def eds_score(df):
    if df['Entegre Derece Skor'] == 12: return 10
    elif df['Entegre Derece Skor'] == 11: return 8
    elif df['Entegre Derece Skor'] == 10: return 6
    elif df['Entegre Derece Skor'] == 9: return 4
    elif df['Entegre Derece Skor'] == 8: return 3
    elif df['Entegre Derece Skor'] == 7: return 2
    elif df['Entegre Derece Skor'] == 6: return 1
    return 0

def yz_score(df):
    if df['Yapay Zeka Risk Sınıfı'] == 5: return 10 + df['Gecikme'] * 6
    elif df['Yapay Zeka Risk Sınıfı'] == 4: return 8 + df['Gecikme'] * 4
    elif df['Yapay Zeka Risk Sınıfı'] == 3: return 6 + df['Gecikme'] * 2
    elif df['Yapay Zeka Risk Sınıfı'] == 2: return 4 + df['Gecikme'] * 1
    return 0

def kur_riski_score(df):
    if df['Kur Riski Çalışması Sınıfı'] == 'Yüksek Riskli': return 4
    elif df['Kur Riski Çalışması Sınıfı'] == 'Yakından İzlenmeli': return 2
    return 0

def kvfb_score(df):
    if df['KVFB / Net Satışlar'] > 2: return 2
    elif df['KVFB / Net Satışlar'] > 1: return 1
    return 0

def borclanma_score(df):
    if df['Borçlanma Oranı'] > 8 or df['YENI_OZSERMAYE'] < 0: return 2
    elif df['Borçlanma Oranı'] > 4: return 1
    return 0

def kaldirac_score(df):
    riskli_sektor_list = [
        'İNŞAAT VE TAAHHÜT',
        'BELEDİYELER VE OSBLER',
        'MADENCİLİK',
        'GEMİCİLİK VE TERSANECİLİK',
        'TURİZM',
    ]
    if df['Tahsis Sektör Adı'] in riskli_sektor_list:
        low, high = 4, 12
    else:
        low, high = 8, 14
    if df['Kaldıraç Oranı'] > high or df['EBITDA'] < 0: return 2
    elif df['Kaldıraç Oranı'] > low: return 1
    return 0

def faiz_yuku_score(df):
    if df['Faiz Yükü Karşılama Farkı'] < -6: return 2
    elif df['Faiz Yükü Karşılama Farkı'] < 0: return 1
    return 0

def cari_oran_score(df):
    if df['Cari Oran'] < 1: return 2
    elif df['Cari Oran'] < 2: return 1
    return 0

def sektor_score(df):
    if df['Sektör Derecesi'] in ['C+', 'C', 'C-', 'D+', 'D', 'D-']: return 6
    elif df['Sektör Derecesi'] == 'B-': return 3
    return 0

def gecikme_score(df):
    if df['Gecikme Gün'] >= 20: return 7
    elif df['Gecikme Gün'] >= 10: return 5
    elif df['Gecikme Gün'] >= 1: return 3
    return 0

def mdr_score(df):
    if df['TSPS İflas vb'] or df['Diğer Banka Takip'] or df['Diğer Banka Tahakkuk'] or df['Diğer Banka Tazmin']:
        return np.inf
    elif df['Gecikme Gün'] >= 10: return np.inf
    return 0

def mdr_score_gecikme(df):
    if df['TSPS İflas vb'] or df['Diğer Banka Takip'] or df['Diğer Banka Tahakkuk'] or df['Diğer Banka Tazmin']:
        return np.inf
    elif df['Gecikme Gün'] >= 20: return np.inf
    return 0

def kroa_score(df):
    if df['KRÖA Yaklaşma']: return 4
    return 0

def limit_doluluk_score_9095(df):
    if df['Sektör Limit Doluluk Oranı'] >= 0.95 or df['Sektör Nakdi Limit Doluluk Oranı'] >= 0.95:
        return 4
    elif df['Sektör Limit Doluluk Oranı'] >= 0.90 or df['Sektör Nakdi Limit Doluluk Oranı'] >= 0.90:
        return 2
    return 0

def limit_doluluk_score_8090(df):
    if df['Sektör Limit Doluluk Oranı'] >= 0.90 or df['Sektör Nakdi Limit Doluluk Oranı'] >= 0.90:
        return 4
    elif df['Sektör Limit Doluluk Oranı'] >= 0.80 or df['Sektör Nakdi Limit Doluluk Oranı'] >= 0.80:
        return 2
    return 0

def limit_doluluk_score_8090_36(df):
    if df['Sektör Limit Doluluk Oranı'] >= 0.90 or df['Sektör Nakdi Limit Doluluk Oranı'] >= 0.90:
        return 6
    elif df['Sektör Limit Doluluk Oranı'] >= 0.80 or df['Sektör Nakdi Limit Doluluk Oranı'] >= 0.80:
        return 3
    return 0

def limit_doluluk_score_8090_48(df):
    if df['Sektör Limit Doluluk Oranı'] >= 0.90 or df['Sektör Nakdi Limit Doluluk Oranı'] >= 0.90:
        return 8
    elif df['Sektör Limit Doluluk Oranı'] >= 0.80 or df['Sektör Nakdi Limit Doluluk Oranı'] >= 0.80:
        return 4
    return 0

def blimit_doluluk_score_8090(df):
    if df['Bankamız Limit Doluluk Oranı'] >= 0.90 or df['Bankamız Nakdi Limit Doluluk Oranı'] >= 0.90:
        return 4
    elif df['Bankamız Limit Doluluk Oranı'] >= 0.80 or df['Bankamız Nakdi Limit Doluluk Oranı'] >= 0.80:
        return 2
    return 0

def limit_doluluk_score_any(df):
    s, b, n = 'Sektör ', 'Bankamız ', 'Nakdi '
    ldo = 'Limit Doluluk Oranı'
    sldo = s + ldo
    snldo = s + n + ldo
    bldo = b + ldo
    bnldo = b + n + ldo
    low, high = 0.8, 0.9
    if any(df[x] >= high for x in [sldo, snldo, bldo, bnldo]): return 4
    elif any(df[x] >= low for x in [sldo, snldo, bldo, bnldo]): return 2
    return 0

In [36]:
puan_column_list = [
    'Finansal Güçlük Puanı', 'Entegre Derece Puanı',
    'Risk Sınıfı Puanı', 'Kur Riski Puanı',
    'KVFB / Net Satışlar Puanı', 'Borçlanma Oranı Puanı',
    'Kaldıraç Oranı Puanı', 'Faiz Yükü Karşılama Puanı', 'Cari Oran Puanı',
    'Sektör Riskliliği Puanı', 'Gecikme Puanı', 'KRÖA Yaklaşma Puanı',
    'Öncül Risklilik Puanı (D.B. Tahakkuk, Takip, Tazmin, İflas, 10+ Gecikme)'
]

puan_fun_list = [
    fg_score, eds_score,
    yz_score, kur_riski_score,
    kvfb_score, borclanma_score,
    kaldirac_score, faiz_yuku_score, cari_oran_score,
    sektor_score, gecikme_score, kroa_score,
    mdr_score
]

puan_column_list_gecikme = puan_column_list

puan_fun_list_gecikme = [
    fg_score, eds_score,
    yz_score, kur_riski_score,
    kvfb_score, borclanma_score,
    kaldirac_score, faiz_yuku_score, cari_oran_score,
    sektor_score, gecikme_score, kroa_score,
    mdr_score_gecikme
]

puan_column_list_mali = [
    'Finansal Güçlük Puanı', 'Entegre Derece Puanı',
    'Risk Sınıfı Puanı', 'Kur Riski Puanı',
    'Sektör Riskliliği Puanı', 'Gecikme Puanı', 'KRÖA Yaklaşma Puanı',
    'Öncül Risklilik Puanı (D.B. Tahakkuk, Takip, Tazmin, İflas, 10+ Gecikme)'
]

puan_fun_list_mali = [
    fg_score, eds_score,
    yz_score, kur_riski_score,
    sektor_score, gecikme_score, kroa_score,
    mdr_score
]

puan_column_list_mali_gecikme = puan_column_list_mali

puan_fun_list_mali_gecikme = [
    fg_score, eds_score,
    yz_score, kur_riski_score,
    sektor_score, gecikme_score, kroa_score,
    mdr_score_gecikme
]

puan_column_list_mali_ldo_9095 = [
    'Finansal Güçlük Puanı', 'Entegre Derece Puanı',
    'Risk Sınıfı Puanı', 'Kur Riski Puanı',
    'Sektör Riskliliği Puanı', 'Gecikme Puanı', 'KRÖA Yaklaşma Puanı',
    'Limit Doluluk Puanı',
    'Öncül Risklilik Puanı (D.B. Tahakkuk, Takip, Tazmin, İflas, 10+ Gecikme)'
]

puan_fun_list_mali_ldo_9095 = [
    fg_score, eds_score,
    yz_score, kur_riski_score,
    sektor_score, gecikme_score, kroa_score,
    limit_doluluk_score_9095,
    mdr_score
]

puan_column_list_mali_ldo_8090 = puan_column_list_mali_ldo_9095

puan_fun_list_mali_ldo_8090 = [
    fg_score, eds_score,
    yz_score, kur_riski_score,
    sektor_score, gecikme_score, kroa_score,
    limit_doluluk_score_8090,
    mdr_score
]

puan_column_list_mali_ldo_8090_36 = puan_column_list_mali_ldo_9095
puan_column_list_mali_ldo_8090_48 = puan_column_list_mali_ldo_9095

puan_fun_list_mali_ldo_8090_36 = [
    fg_score, eds_score,
    yz_score, kur_riski_score,
    sektor_score, gecikme_score, kroa_score,
    limit_doluluk_score_8090_36,
    mdr_score
]
puan_fun_list_mali_ldo_8090_48 = [
    fg_score, eds_score,
    yz_score, kur_riski_score,
    sektor_score, gecikme_score, kroa_score,
    limit_doluluk_score_8090_48,
    mdr_score
]

puan_column_list_mali_bldo_8090 = [
    'Finansal Güçlük Puanı', 'Entegre Derece Puanı',
    'Risk Sınıfı Puanı', 'Kur Riski Puanı',
    'Sektör Riskliliği Puanı', 'Gecikme Puanı', 'KRÖA Yaklaşma Puanı',
    'Sektör Limit Doluluk Puanı', 'Bankamız Limit Doluluk Puanı',
    'Öncül Risklilik Puanı (D.B. Tahakkuk, Takip, Tazmin, İflas, 10+ Gecikme)'
]

puan_fun_list_mali_bldo_8090 = [
    fg_score, eds_score,
    yz_score, kur_riski_score,
    sektor_score, gecikme_score, kroa_score,
    limit_doluluk_score_8090, blimit_doluluk_score_8090,
    mdr_score
]

puan_column_list_mali_ldo_any = [
    'Finansal Güçlük Puanı', 'Entegre Derece Puanı',
    'Risk Sınıfı Puanı', 'Kur Riski Puanı',
    'Sektör Riskliliği Puanı', 'Gecikme Puanı', 'KRÖA Yaklaşma Puanı',
    'Sektör/Banka Limit Doluluk Puanı',
    'Öncül Risklilik Puanı (D.B. Tahakkuk, Takip, Tazmin, İflas, 10+ Gecikme)'
]

puan_fun_list_mali_ldo_any = [
    fg_score, eds_score,
    yz_score, kur_riski_score,
    sektor_score, gecikme_score, kroa_score,
    limit_doluluk_score_any,
    mdr_score
]

def create_pra_analysis(df_pra, puan_column_list=puan_column_list, puan_fun_list=puan_fun_list, threshold_list=None):
    df = df_pra.copy()
        
    for c, f in zip(puan_column_list, puan_fun_list):
        df[c] = df.apply(f, axis=1)

    df['Toplam Puan'] = 0
    for c in puan_column_list:
        df['Toplam Puan'] += df[c]

    dfo = create_pra_risk_categories(df, 'Toplam Puan', threshold_list)
    df = pd.merge(df, dfo[['Toplam Puan', 'Risk Kategorisi']], on='Toplam Puan', how='left')

    df['Takıldığı Toplam Kriter Sayısı'] = 0
    for c in puan_column_list:
        df['Takıldığı Toplam Kriter Sayısı'] += df[c].apply(lambda x: min(1, x))

    # dfyr = df.copy()
    # dfyr = dfyr.loc[dfyr['Risk Kategorisi'].isin(['Yüksek', 'Çok Yüksek'])]

    # df = pd.merge(df, dfyr.loc[:, ['Müşteri No'] + [x for x in list(dfyr.columns) if x not in list(df.columns)]], on='Müşteri No', how='left')
    # dfyr = dfyr.loc[dfyr['Risk Kategorisi'].isin(['Çok Yüksek', 'Yüksek'])]

    # return df.copy(), dfyr.copy(), dfo.copy()
    return df.copy()

# Analiz Kümesi

In [31]:
df_master = df_old.copy()

c0 = 'Bankamız Ticari Portföy ({})'.format(last_date)
mn = 'Müşteri No'
nr = 'Nakdi Reeskontlu Toplam'
gnr = 'G.Nakdi Toplam'
tr = 'Toplam Reeskontlu Risk'
c_list = ['Küme', 'Adet', 'Nakdi Risk', 'G.Nakdi Risk', 'Toplam Risk']

df = df_master.copy()
d = 'Çıkarılma Sebebi'
dfd = pd.DataFrame(columns=list(df_master.columns) + [d])
dfk = pd.DataFrame(columns=c_list)

c = c0
dfk.loc[len(dfk)] = return_excluded_info(c, df, False)


pra_bolum_exclude_list = ['Krediler İzleme Bölümü', 'Ticari ve Kurumsal Krediler Takip Bölümü']
c = 'Yakın İzleme / 2. Sınıf'
dfe = df.loc[(df['Müşteri Sınıfı'] == 2) | (df['İlgili Bölüm'] == 'Krediler İzleme Bölümü')]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)

c = 'Takip / 3. Sınıf'
dfe = df.loc[(df['Müşteri Sınıfı'] >= 3) | (df['İlgili Bölüm'] == 'Ticari ve Kurumsal Krediler Takip Bölümü')]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)

df = df.loc[(df['Müşteri Sınıfı'] == 1) & (~df['İlgili Bölüm'].isin(pra_bolum_exclude_list))]


dfk = add_h_space(dfk)

c = 'Banka Risk Grubu'
dfe = df.loc[(df['Risk Grubu Kodu'] == 'BANKRG')]
dfd = df.loc[(df['Risk Grubu Kodu'] != 'BANKRG')]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)

c = 'Finans'
dfe = df.loc[(df['Tahsis Sektör Adı'] == 'FİNANS')]
df = df.loc[(df['Tahsis Sektör Adı'] != 'FİNANS')]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)

c = 'PPP (Hastane ve Otoyollar)'
dfe = df.loc[(df['Tahsis Sektör Adı'] == 'PPP (HASTANE VE OTOYOLLAR)')]
df = df.loc[(df['Tahsis Sektör Adı'] != 'PPP (HASTANE VE OTOYOLLAR)')]
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)

c = 'Belediyeler ve OSBler'
dfe = df.loc[(df['Tahsis Sektör Adı'] == 'BELEDİYELER VE OSBLER')]
df = df.loc[(df['Tahsis Sektör Adı'] != 'BELEDİYELER VE OSBLER')]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)



c = 'Kıbrıs Şubesi'
dfe = df.loc[(df['Şube Türü'] == 'Kıbrıs')]
df = df.loc[(df['Şube Türü'] != 'Kıbrıs')]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)

c = 'Yurt Dışı Şubesi'
dfe = df.loc[(df['Şube Türü'] == 'Yurt Dışı')]
df = df.loc[(df['Şube Türü'] != 'Yurt Dışı')]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)

c = 'Serbest Bölge Şubesi'
dfe = df.loc[(df['Şube Türü'] == 'Serbest Bölge')]
df = df.loc[(df['Şube Türü'] != 'Serbest Bölge')]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)

c = 'Özel Bankacılık Şubesi'
dfe = df.loc[(df['Şube Türü'] == 'Özel Bankacılık')]
df = df.loc[(df['Şube Türü'] != 'Özel Bankacılık')]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)


c = 'Son Ay Risk Skoru Eksik'
dfe = df.loc[(df['Yapay Zeka Risk Sınıfı'].isnull())]
df = df.loc[(df['Yapay Zeka Risk Sınıfı'].notnull())]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)

c = 'Son Ay Derecelendirme Eksik'
dfe = df.loc[(df['Entegre Derece Skor'].isnull())]
df = df.loc[(df['Entegre Derece Skor'].notnull())]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)

# c = 'Mali Tablosu Eksik'
# dfe = df.loc[(df['FST_YR'].isnull())]
# df = df.loc[(df['FST_YR'].notnull())]
# dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# # dfe[d] = c
# # dfd = pd.concat([dfd, dfe], ignore_index=True)


# # risk_lower = 1_000_000
# risk_upper = np.inf
# risk_lower = 100_000
# # risk_upper = 1_000_000
# c = f'Toplam Risk < {risk_lower:,} veya >= {risk_upper:,} TL'

# dfe = df.loc[(df['Toplam Reeskontlu Risk'] >= risk_upper) | (df['Toplam Reeskontlu Risk'] < risk_lower)]
# df = df.loc[(df['Toplam Reeskontlu Risk'] < risk_upper) & (df['Toplam Reeskontlu Risk'] >= risk_lower)]
# dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# # dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)


# pra_segment_exclude_list = ['KURUMSAL', 'TİCARİ', 'İŞLETME']
pra_segment_exclude_list = ['KURUMSAL', 'TİCARİ', 'KOBİ']
# pra_segment_exclude_list = ['İŞLETME', 'KOBİ']
c = ', '.join([x.capitalize() for x in pra_segment_exclude_list])
dfe = df.loc[(df['Değerlendirmeye Esas Segment'].isin(pra_segment_exclude_list))]
df = df.loc[(~df['Değerlendirmeye Esas Segment'].isin(pra_segment_exclude_list))]
dfk.loc[len(dfk)] = return_excluded_info(c, dfe)
# dfe[d] = c
# dfd = pd.concat([dfd, dfe], ignore_index=True)


dfx = dfk.copy().dropna()
neg_sums = [np.sum([min(x, 0) for x in dfx[x]]) for x in dfx.columns[1:]]
dfk.loc[len(dfk)] = ['Toplam Hariç Tutulan Küme'] + neg_sums
dfk = add_h_space(dfk)

c = 'İncelenen Firmalar'
dfe = df.copy()
dfk.loc[len(dfk)] = return_excluded_info(c, dfe, False)
dfk.loc[len(dfk)] = ['İncelenen Firmalar/Ticari Portföy'] + return_percentages('İncelenen Firmalar')[1:]

dfk.loc[len(dfk)] = [f'{last_date} KR202, Reeskontlu, mio TL'] + [np.nan] * (len(dfk.columns) - 1)
kcl = list(dfk.columns)
kcl = kcl[0:1] + [np.nan] + kcl[1:]
dfk[np.nan] = np.nan
dfk = dfk[kcl]

# df_pra_kume = insert_corner(dfk)
# dfk = dfk.fillna('')
df_pra_kume = dfk.copy()
df_pra_cikarilan = dfd.copy()
df_pra_backup = df.copy()

df_pra_kume

In [32]:
# quick_export(df_pra_kume, 'ktkume')

In [33]:
df_pra = create_pra_analysis(df_pra_backup)
df_pra_pass = create_pra_pass_alt(df_pra, 'Risk Kategorisi', risk_category_list)
df_pra_pass

,Risk Kategorisi,Adet,%,NaN,2. Sınıfa Geçen Adet,2. Sınıfa Geçiş %,Ortalama 2. Sınıfa Geçiş Ayı,NaN,3. Sınıfa Geçen Adet,3. Sınıfa Geçiş %,Ortalama 3. Sınıfa Geçiş Ayı,NaN,Riski Kapanan,Riski Kapanma %
0,Mükemmel,"123,154.00",0.33,,"2,625.00",0.02,7.52,,197.00,0.00,8.80,,"4,168.00",0.03
1,Çok Düşük,"45,537.00",0.12,,"1,849.00",0.04,7.32,,79.00,0.00,8.01,,"1,365.00",0.03
2,Düşük,"69,783.00",0.19,,"4,864.00",0.07,7.29,,246.00,0.00,9.06,,"2,120.00",0.03
3,Orta,"88,476.00",0.24,,"13,113.00",0.15,6.63,,736.00,0.01,8.62,,"3,164.00",0.04
4,Yüksek,"23,667.00",0.06,,"6,346.00",0.27,5.62,,403.00,0.02,8.57,,"1,195.00",0.05
5,Çok Yüksek,"17,188.00",0.05,,"6,470.00",0.38,4.61,,501.00,0.03,6.82,,"1,651.00",0.10
6,,,,,,,,,,,,,,
7,Genel Toplam,"367,805.00",1.00,,"35,267.00",0.10,6.27,,"2,162.00",0.01,8.24,,"13,663.00",0.04


In [116]:
# dfp = create_pra_analysis(df_pra_backup, puan_column_list_mali_bldo_8090, puan_fun_list_mali_bldo_8090)
# df_pra_puan_pass = create_pra_pass_alt(dfp, 'Toplam Puan', sorted(dfp['Toplam Puan'].unique()))
# quick_export(df_pra_puan_pass, 'İşletme Puan '+ now())

In [117]:
# quick_export(dfp.loc[(dfp['Risk Kategorisi'] == 'Mükemmel') & (dfp['Müşteri Sınıfı Max'] == 2)].select_dtypes(exclude='O').describe(), 'BLDO8090')

In [38]:
threshold_list = [0, 2, 5, 10, 15, np.inf]
# threshold_list = [0, 1, 3, 6, 11, np.inf]

df_pra_2 = create_pra_analysis(df_pra_backup, puan_column_list_mali_ldo_8090_48, puan_fun_list_mali_ldo_8090_48, threshold_list)
df_pra_pass_2 = create_pra_pass_alt(df_pra_2, 'Risk Kategorisi', risk_category_list)

df1 = insert_header(df_pra_pass, 'Mali Tablo Var, 10+ Gecikme = Çok Yüksek')
df2 = insert_header(df_pra_pass_2, 'Mali Tablo Var, 20+ Gecikme = Çok Yüksek')

if threshold_list is not None:
    labels = ['0'] + [f'{x + 1} ve Üzeri' if y == np.inf else f'{x + 1} - {y}' if x + 1 != y else str(x + 1) for x, y in zip(threshold_list[:], threshold_list[1:])]
    df = pd.DataFrame()
    df['Toplam Puan'] = labels
    df['Risk Kategorisi'] = risk_category_list
    df = insert_header(df, 'Puanlama')
    df2 = h_stack([df2, df], 1)

dff = insert_corner(v_stack([df1, df2]))
quick_export(dff, 'İşletme' + now())

PermissionError: [Errno 13] Permission denied: 'C:\\Users\\105617\\Desktop\\Workspace\\Panel\\output\\İşletme03251141.xlsx'

In [56]:
import seaborn as sns
import matplotlib.pyplot as plt

show_one_half = False
save_figure = True
rearrange = True
correlation_matrix_file_name  ='PRA {} {}2.png'.format(pra_v, last_date)

df = df_pra_2.copy()
df = df.loc[df['Risk Kategorisi'] == 'Mükemmel']
df = df.select_dtypes(exclude='O')
correlation_exclude_list = ['Müşteri No', 'Müşteri Sınıfı', 'Geçiş Ayı', 'Karşılık', 'Şube', 'FST_YR', 'FST_MO']
x = ['Müşteri Sınıfı Max'] + [x for x in df.columns if not any(y in x for y in correlation_exclude_list)]

df = df[x]
dfcm = df.corr()
dfcm = dfcm.dropna(how='all', axis=0)
dfcm = dfcm.dropna(how='all', axis=1)
dfcm = dfcm.sort_values('Müşteri Sınıfı Max', ascending=False)
dfcm = dfcm[dfcm.index]


upper_half = np.triu(dfcm) if show_one_half else None

fig = plt.figure(figsize=(60, 60), dpi = 350)
sns.heatmap(dfcm, annot=True, fmt='.2f', square=True, mask=upper_half)

if save_figure:
    plt.savefig(output_folder_path + correlation_matrix_file_name, bbox_inches = 'tight')
    plt.close()

In [100]:
quick_export([insert_corner(df_pra_kume), insert_corner(df_pra_pass)], 'PRA-3', ['Küme', 'Geçişler'])

In [89]:
create_pra_pass_alt(df_pra_mali, 'Toplam Puan', sorted(df_pra_mali['Toplam Puan'].unique()))

,Toplam Puan,Adet,NaN,2. Sınıfa Geçen Adet,2. Sınıfa Geçiş %,Ortalama 2. Sınıfa Geçiş Ayı,NaN,3. Sınıfa Geçen Adet,3. Sınıfa Geçiş %,Ortalama 3. Sınıfa Geçiş Ayı,NaN,Riski Kapanan,Riski Kapanma %
0,0.00,"25,723.00",,379.00,0.01,7.77,,24.00,0.00,9.50,,303.00,0.01
1,1.00,939.00,,45.00,0.05,7.20,,1.00,0.00,5.00,,30.00,0.03
2,2.00,712.00,,48.00,0.07,7.27,,3.00,0.00,8.00,,54.00,0.08
3,3.00,497.00,,34.00,0.07,6.97,,,,,,8.00,0.02
4,4.00,"7,380.00",,440.00,0.06,7.71,,14.00,0.00,8.43,,97.00,0.01
5,5.00,999.00,,113.00,0.11,6.82,,5.00,0.01,8.00,,18.00,0.02
6,6.00,"4,119.00",,465.00,0.11,7.15,,28.00,0.01,9.21,,94.00,0.02
7,7.00,939.00,,154.00,0.16,6.68,,15.00,0.02,10.73,,31.00,0.03
8,8.00,"1,845.00",,362.00,0.20,6.83,,21.00,0.01,9.57,,56.00,0.03
9,9.00,499.00,,129.00,0.26,6.05,,8.00,0.02,8.50,,17.00,0.03


In [23]:
create_pra_summary(df_pra)

,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,NaN,Risk Kategorisi,NaN,Adet,Adet Payı,NaN,Nakdi Risk,Nakdi Risk Payı,NaN,Ortalama Entegre Derece / Skor,Ortalama Yapay Zeka Risk Sınıfı
1,NaN,Mükemmel,NaN,"2,001.00",0.11,NaN,741.56,0.10,NaN,2.51,1.00
2,NaN,Çok Düşük,NaN,"5,669.00",0.31,NaN,"2,262.22",0.29,NaN,3.30,1.00
3,NaN,Düşük,NaN,"5,667.00",0.31,NaN,"2,491.34",0.32,NaN,4.30,1.05
4,NaN,Orta,NaN,"3,600.00",0.20,NaN,"1,643.99",0.21,NaN,5.39,1.57
5,NaN,Yüksek,NaN,937.00,0.05,NaN,453.91,0.06,NaN,7.04,2.41
6,NaN,Çok Yüksek,NaN,306.00,0.02,NaN,134.57,0.02,NaN,7.92,2.73
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,Genel Toplam,NaN,"18,180.00",1.00,NaN,"7,727.59",1.00,NaN,4.21,1.23
9,NaN,"2023.12 KR202, Reeskontlu, mio TL",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Model Karşılaştırma

In [47]:
def create_pra_model_comp(df_pra_max):
    dfp = df_pra_max.copy()
    rk_list = ['Düşük Riskli', 'Yüksek Riskli']
    rk_upper = ['Yüksek', 'Çok Yüksek']
    rk = 'Risk Kategorisi'
    mn = 'Müşteri No'
    nrt = 'Nakdi Reeskontlu Toplam'
    msm = 'Müşteri Sınıfı Max'
    factor = 1_000_000

    dfv_list_list_list = []
    for find_risk, c in zip([False, True], [mn, nrt]):
        dfv_list_list = []
        for ss in [2, 3]:
            dfv_list = []
            for x, y, th in zip(['Entegre Derece Skor', 'Yapay Zeka Risk Sınıfı'], ['Entegre Derece / Skor', 'Yapay Zeka Risk Sınıfı'], [[(1, 7), (8, 12)], [(1, 2), (3, 5)]]):
                dfa = dfp.loc[dfp[msm] == ss, [rk, c, x]]
                df = pd.DataFrame()
                df[rk] = rk_list
                df = add_v_space(df)
                t_list = ['Genel Toplam', np.nan]
                for lower, upper in th:
                    n = f'{lower} - {upper}'
                    dft = dfa.loc[(dfa[x] >= lower) & (dfa[x] <= upper), [rk, c]]
                    if find_risk:
                        u = dft.loc[dfp[rk].isin(rk_upper), c].sum()
                        t = dft[c].sum()
                    else:
                        u = dft.loc[dfp[rk].isin(rk_upper), c].count()
                        t = dft[c].count()
                    t_list.append(t)
                    l = t - u
                    df[n] = [l, u]

                
                df = add_v_space(df)
                t_list.append(np.nan)

                n = 'Genel Toplam'
                if find_risk:
                    u = dfa.loc[dfa[rk].isin(rk_upper), c].sum()
                    t = dfa[c].sum()
                else:
                    u = dfa.loc[dfa[rk].isin(rk_upper), c].count()
                    t = dfa[c].count()
                t_list.append(t)
                l = t - u
                df[n] = [l, u]
                
                df = add_h_space(df)
                df.loc[len(df)] = t_list
                if find_risk:
                    df.iloc[:, 1:] /= factor
                    

                header_end = 'Nakdi Risk' if find_risk else 'Müşteri Adedi'
                header = f'{ss}. Sınıfa Geçen {header_end}'
                dfx = pd.DataFrame(columns=[header, np.nan, y])
                df = v_stack([dfx, df], 0)
                footer_end = 'Reeskontlu, mio TL' if find_risk else 'Adet'
                footer = f'{start_term}-{end_term} KR202, {footer_end}'
                df.loc[len(df)] = [footer] + [np.nan] * (len(df.columns) - 1)
                df.loc[len(df)] = ['Riski kapanan firmalar dahildir'] + [np.nan] * (len(df.columns) - 1)

                dfv_list.append(df)
            dfv_list_list.append(h_stack(dfv_list, 4))
        dfv_list_list_list.append(v_stack(dfv_list_list))

    return insert_corner(v_stack(dfv_list_list_list, 6))

## Değişim

### Alt

In [14]:
def create_pra_dist_alt(df_pra, rows, row_list, use_thresholds=False, reorder=False, rename_dict=None, factor=1_000_000, find_risk=False):
    n_list = ['Mükemmel', 'Çok Düşük', 'Düşük', 'Orta', 'Yüksek', 'Çok Yüksek']
    rk = 'Risk Kategorisi'
    yrc = 'Yüksek Riskli Portföy Oranı'
    mn = 'Müşteri No'
    nrt = 'Nakdi Reeskontlu Toplam'
    c = rows
    c_list = row_list
    dfp = df_pra.copy()
    x = nrt if find_risk else mn

    df = pd.DataFrame()
    if use_thresholds:
        ettl = c_list
        etl = [f'{et[0]} - {et[1]}' for et in ettl]
        df[c] = etl
    else:
        df[c] = c_list if c_list is not None else sorted(df_pra.loc[df_pra[c].notnull(), c].unique())
        
    t_list = ['Genel Toplam']

    for n in n_list:
        dft = dfp.copy()
        if find_risk:
            dft = dft.loc[dft[rk] == n, [x, c]].groupby(c).sum().reset_index().rename(columns={x: n})
        else:
            dft = dft.loc[dft[rk] == n, [x, c]].groupby(c).count().reset_index().rename(columns={x: n})
        if use_thresholds:
            nx_list = []
            for et in ettl:
                nx = dft.loc[(dft[c] >= et[0]) & (dft[c] <= et[1]), n].sum()
                nx_list.append(nx)
            df[n] = nx_list
        else:
            df = pd.merge(df, dft, on=c, how='outer')
        t_list.append(df[n].sum())

    n = 'Toplam'
    dft = dfp.copy()
    if find_risk:
        dft = dft[[x, c]].groupby(c).sum().reset_index().rename(columns={x: n})
    else:
        dft = dft[[x, c]].groupby(c).count().reset_index().rename(columns={x: n})
    if use_thresholds:
        nx_list = []
        for et in ettl:
            nx = dft.loc[(dft[c] >= et[0]) & (dft[c] <= et[1]), n].sum()
            nx_list.append(nx)
        df[n] = nx_list
    else:
        df = pd.merge(df, dft, on=c, how='outer')
    t_list.append(df[n].sum())

    df = df.dropna(axis=0, how='all', subset=list(df.columns)[1:], ignore_index=True)
    if rename_dict is not None:
        df[c] = df[c].map(rename_dict)
    df = add_h_space(df)
    df.loc[len(df)] = t_list
    if find_risk:
        df.iloc[:,1:] = df.iloc[:,1:] / factor
    df[yrc] = df[['Yüksek', 'Çok Yüksek']].sum(axis=1) / df['Toplam']
    if reorder:
        df.iloc[:-1, :] = df.iloc[:-1, :].sort_values(yrc, ascending=False)


    return df[[c, yrc]]

### Değişim

In [15]:
def create_pra_dist_change(df_old, df_new, rows, row_list, row_new=None, sort=0, use_thresholds=False, drop_na=True, study_scope=None):
    dfn = df_new.copy()
    dfo = df_old.copy()

    yrpo = 'Yüksek Riskli Portföy Oranı'

    r = rows
    r_list = row_list
    r_dict = row_new
    
    dfv_list = []

    for find_risk in [False, True]:
        df = pd.DataFrame()
        if use_thresholds:
            df[r] = [f'{et[0]} - {et[1]}' for et in r_list] + ['Genel Toplam']
        else:
            df[r] = r_list + ['Genel Toplam']
        df = add_v_space(df)

        dft = create_pra_dist_alt(dfo, r, r_list, use_thresholds, find_risk=find_risk)
        dft.columns = [r, start_term]
        df = pd.merge(df, dft, on=r, how='left')
        dft = create_pra_dist_alt(dfn, r, r_list, use_thresholds, find_risk=find_risk)
        dft.columns = [r, end_term]
        df = pd.merge(df, dft, on=r, how='left')

        if drop_na:
            df = df.dropna(subset=list(df.columns)[2:], how='all').reset_index(drop=True)

        df['Değişim'] = df[end_term] - df[start_term]
        

        if r_dict is not None:
            if type(r_dict) is list:
                r_dict = dict(zip(r_list, r_dict))
            df[r] = df[r].apply(lambda x: r_dict.get(x, x))

        if sort is not None:
            df.loc[df[end_term] == 0, 'Değişim'] = df.loc[df[end_term] == 0, start_term] * -100
            df.loc[(df[end_term] == 0) & (df[start_term] == 0), 'Değişim'] = -1_000_000
            if type(sort) is str:
                if sort.startswith('-'):
                    df[:-1] = df[:-1].sort_values(sort[1:], ascending=False)
                else:
                    df[:-1] = df[:-1].sort_values(sort)
            else:
                if sort > 0:
                    df[:-1] = df[:-1].sort_values('Değişim')
                elif sort < 0:
                    df[:-1] = df[:-1].sort_values('Değişim', ascending=False)
            df['Değişim'] = df[end_term] - df[start_term]

        df = insert_row(df, -1)

        header = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
        df = insert_header(df, header)
        df = insert_row(df, 0, [np.nan, np.nan, yrpo])
        footer_end = 'Reeskontlu' if find_risk else 'Adet'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        df.loc[len(df)] = [footer] + [np.nan] * (len(df.columns) - 1)
        
        if study_scope is not None:
            footer_scope = f'{start_term} ve {end_term} çalışma kümelerine {study_scope[0]} ve üzeri müşterisi giren {study_scope[1]} dikkate alınmıştır'
            df.loc[len(df)] = [footer_scope] + [np.nan] * (len(df.columns) - 1)
            df = add_h_space(df)

        dfv_list.append(df.copy())

    return insert_corner(v_stack(dfv_list))

### Scope

In [16]:
def create_pra_dist_change_scope(df_old, df_new, rows, row_list, row_new=None, use_thresholds=False, scope=20, study_scope=(30, 'şubeler')):
    dfn = df_new.copy()
    dfo = df_old.copy()

    yrpo = 'Yüksek Riskli Portföy Oranı'
    r = rows
    r_list = row_list
    r_dict = row_new

    dfv_list = []

    for find_risk in [False, True]:
        df = pd.DataFrame()
        if use_thresholds:
            df[r] = [f'{et[0]} - {et[1]}' for et in r_list] + ['Genel Toplam']
        else:
            df[r] = r_list + ['Genel Toplam']
        df = add_v_space(df)

        dft = create_pra_dist_alt(dfo, r, r_list, use_thresholds, find_risk=find_risk)
        dft.columns = [r, start_term]
        df = pd.merge(df, dft, on=r, how='left')
        dft = create_pra_dist_alt(dfn, r, r_list, use_thresholds, find_risk=find_risk)
        dft.columns = [r, end_term]
        df = pd.merge(df, dft, on=r, how='left')
        df['Değişim'] = df[end_term] - df[start_term]

        if r_dict is not None:
            if type(r_dict) is list:
                r_dict = dict(zip(r_list, r_dict))
            df[r] = df[r].apply(lambda x: r_dict.get(x, x))

        df.loc[df[end_term] == 0, 'Değişim'] = df.loc[df[end_term] == 0, start_term] * -100
        df.loc[(df[end_term] == 0) & (df[start_term] == 0), 'Değişim'] = -1_000_000
        
        dfg = df.copy()
        dfg[:-1] = dfg[:-1].sort_values('Değişim')
        dfg = pd.concat([dfg[:scope], dfg[-1:]]).reset_index(drop=True)

        dfb = df.copy()
        dfb[:-1] = dfb[:-1].sort_values('Değişim', ascending=False)
        dfb = pd.concat([dfb[:scope], dfb[-1:]]).reset_index(drop=True)

        dfg['Değişim'] = dfg[end_term] - dfg[start_term]
        dfb['Değişim'] = dfb[end_term] - dfb[start_term]
        dfg = insert_row(dfg, -1)
        dfb = insert_row(dfb, -1)

        header_end = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
        footer_end = 'Reeskontlu' if find_risk else 'Adet'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        header_good = f'İyileşen {scope}, {header_end}'
        header_bad = f'Kötüleşen {scope}, {header_end}'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        footer_scope = f'{start_term} ve {end_term} çalışma kümelerine {study_scope[0]} ve üzeri müşterisi giren {study_scope[1]} dikkate alınmıştır'

        dfg = insert_header(dfg, header_good)
        dfg = insert_row(dfg, 0, [np.nan, np.nan, yrpo])
        dfg.loc[len(dfg)] = [footer] + [np.nan] * (len(dfg.columns) - 1)
        dfg.loc[len(dfg)] = [footer_scope] + [np.nan] * (len(dfg.columns) - 1)
        dfg = add_h_space(dfg)

        dfb = insert_header(dfb, header_bad)
        dfb = insert_row(dfb, 0, [np.nan, np.nan, yrpo])
        dfb.loc[len(dfb)] = [footer] + [np.nan] * (len(dfb.columns) - 1)
        dfb.loc[len(dfb)] = [footer_scope] + [np.nan] * (len(dfb.columns) - 1)
        dfb = add_h_space(dfb)

        dfv_list.append(h_stack([dfb, dfg]))

    return insert_corner(v_stack(dfv_list))

### Adet-Risk Değişimi

In [17]:
def create_pra_count_risk_change(df_old, df_new, rows, row_list, row_new=None, drop_na=True, sort=True, factor=1_000_000):

    dfn = df_new.copy()
    dfo = df_old.copy()

    dfn = dfn.loc[dfn['Risk Kategorisi'].isin(['Çok Yüksek', 'Yüksek'])]
    dfo = dfo.loc[dfo['Risk Kategorisi'].isin(['Çok Yüksek', 'Yüksek'])]
    r = rows
    r_list = row_list
    r_dict = row_new

    dfv_list = []

    for c, nn, pp, find_risk in zip(['Müşteri No', 'Nakdi Reeskontlu Toplam'], ['Adet', 'Nakdi Risk'], ['Adet Payı', 'Nakdi Risk Payı'], [False, True]):
        df = pd.DataFrame()
        df[r] = r_list
        df = add_v_space(df)
        t_list = ['Genel Toplam', np.nan]
        start = start_term + ' ' + pp
        end = end_term + ' ' + pp

        for dfp, d in zip([dfo, dfn], [start_term, end_term]):
            n = d + ' ' + nn
            p = d + ' ' + pp
            dft = dfp[[r, c]].copy()
            if find_risk:
                dft = dft.groupby(r).sum().reset_index().rename(columns={c: n})
                dft[n] /= factor
            else:
                dft = dft.groupby(r).count().reset_index().rename(columns={c: n})
            df = pd.merge(df, dft, on=r, how='outer')
            t = df[n].sum()
            t_list.append(t)
            df[p] = df[n] / t
            t_list.append(1)

        if drop_na:
            df = df.dropna(subset=df.columns[2:], how='all').reset_index(drop=True)

        if r_dict is not None:
            df[r] = df[r].apply(lambda x: r_dict.get(x, x))

        if sort:
            df = df.sort_values([start, end, r], ascending=False).reset_index(drop=True)

        df.iloc[:, 2:] = df.iloc[:, 2:].fillna(0)

        df = add_h_space(df)
        df.loc[len(df)] = t_list

        df['Pay Değişimi'] = df[end] - df[start]
        df.iloc[-1, -1] = np.nan


        header = 'Nakdi Riske Göre' if find_risk else 'Adede Göre'
        df = insert_header(df, 'Yüksek Riskli Portföy İçerisinde')
        df = insert_header(df, header)

        footer_end = 'Reeskontlu, mio TL' if find_risk else 'Adet'
        footer = f'{start_term}-{end_term} KR202, {footer_end}'
        df.loc[len(df)] = [footer] + [np.nan] * (len(df.columns) - 1)

        dfv_list.append(df.copy())

    return insert_corner(v_stack(dfv_list))

# Alt

In [280]:
def create_pra_pass_alt(df_old_max, rows, row_list, row_new=None, find_risk=False, last_date=last_date, drop_na=True):
    r = rows
    r_list = row_list
    r_dict = row_new
    mn = 'Müşteri No'
    nrt = 'Nakdi Reeskontlu Toplam'
    msm = 'Müşteri Sınıfı Max'
    msl = 'Müşteri Sınıfı ' + last_date
    factor = 1_000_000

    if find_risk:
        x = nrt
        s = 'Nakdi Risk'
    else:
        x = mn
        s = 'Adet'
    dfp = df_old_max.copy()
    if find_risk:
        dfp[nrt] /= factor

    t_list = ['Genel Toplam']
    df = pd.DataFrame()
    df[r] = r_list

    c = x
    n = s
    p = '%'
    dft = dfp[[r, c]].copy()
    if find_risk:
        dft = dft.groupby(r).sum().reset_index().rename(columns={c: n})
    else:
        dft = dft.groupby(r).count().reset_index().rename(columns={c: n})
    df = pd.merge(df, dft, on=r, how='left')
    t = df[n].sum()
    t_list.append(t)
    df[p] = df[n] / t
    t_list.append(1)

    df = add_v_space(df)
    t_list.append(np.nan)

    for ss in [2, 3]:
        c = x
        n = f'{ss}. Sınıfa Geçen {s}'
        ga = f'{ss}. Sınıfa Geçiş Ayı'
        dfa = dfp.loc[dfp[msm] == ss, [r, c, ga]].copy()
        if find_risk:
            dft = dfa[[r, c]].groupby(r).sum().reset_index().rename(columns={c: n})
        else:
            dft = dfa[[r, c]].groupby(r).count().reset_index().rename(columns={c: n})
        df = pd.merge(df, dft, on=r, how='left')
        t = df[n].sum()
        t_list.append(t)

        p = f'{ss}. Sınıfa Geçiş %'
        df[p] = df[n] / df[s]
        t = df[n].sum() / df[s].sum()
        t_list.append(t)

        c = f'{ss}. Sınıfa Geçiş Ayı'
        n = f'Ortalama {ss}. Sınıfa Geçiş Ayı'
        t = dfa[ga].mean()
        t_list.append(t)
        dft = dfa[[r, ga]].groupby(r).mean().reset_index().rename(columns={c: n})
        df = pd.merge(df, dft, on=r, how='left')

        df = add_v_space(df)
        t_list.append(np.nan)

    if drop_na:
        df = df.dropna(subset=list(df.columns)[2:], how='all').reset_index(drop=True)

    if r_dict is not None:
        if type(r_dict) is list:
            r_dict = dict(zip(r_list, r_dict))
        df[r] = df[r].apply(lambda x: r_dict.get(x, x))

    df.loc[len(df)] = t_list
    df = df[[r] + [f'{ss}. Sınıfa Geçiş %' for ss in [2, 3]]]
    return df.copy()


## Filtrelenmiş Liste

In [18]:
def get_scope_filtered_list(df_old, df_new, column, filter_value=None, filter_column=None, scope=30):
    dfo = df_old.copy()
    dfn = df_new.copy()
    c = column

    if filter_value is not None:
        x = c if filter_column is None else filter_column
        y = filter_value
        dfo = dfo.loc[dfo[x] == y]
        dfn = dfn.loc[dfn[x] == y]

    dfo = dfo[c].value_counts().reset_index().rename(columns={'count': 'old'})
    dfn = dfn[c].value_counts().reset_index().rename(columns={'count': 'new'})
    df = pd.merge(dfo, dfn, on=c, how='inner')
    filtered_list = list(df.loc[df.apply(lambda x: x['old'] >= scope and x['new'] >= scope, axis=1), c])
    filtered_list.sort(key=locale.strxfrm)

    return filtered_list

# Analiz

In [69]:
df_pra_pass = create_pra_pass(df_pra_old_max, 'Risk Kategorisi', risk_category_list)
df_pra_segment_pass = create_pra_pass(df_pra_old_max, 'Değerlendirmeye Esas Segment', segment_list, segment_title_dict)
df_pra_bolum_pass = create_pra_pass(df_pra_old_max_bf, 'İlgili Bölüm', new_filtered_bolum_list)
df_pra_sektor_pass = create_pra_pass(df_pra_old_max, 'Tahsis Sektör Adı', sorted_sektor_list, sektor_title_dict)
df_pra_sube_pass = create_pra_pass(df_pra_old_max, 'Şube Türü', filtered_sube_turu_list)
df_pra_bolge_pass = create_pra_pass(df_pra_old_max, 'Bölge Adı', bolge_list)
df_pra_model = create_pra_model_comp(df_pra_old_max)

In [70]:
df_pra_segment_change = create_pra_dist_change(df_old, df_new, 'Değerlendirmeye Esas Segment', segment_list, segment_title_list)
df_pra_bolum_change = create_pra_dist_change(df_old_bf, df_new_bf, 'İlgili Bölüm', new_filtered_bolum_list)
df_pra_sube_change = create_pra_dist_change(df_old, df_new, 'Şube Türü', filtered_sube_turu_list)
df_pra_eds_change = create_pra_dist_change(df_old, df_new, 'Entegre Derece Skor', [(1, 5), (6, 7), (8, 12)], use_thresholds=True)
df_pra_bolge_change = create_pra_dist_change(df_old, df_new, 'Bölge Adı', bolge_list, sort=-1)

In [71]:
df_pra_sektor_count_risk_change = create_pra_count_risk_change(df_old, df_new, 'Tahsis Sektör Adı', sorted_sektor_list, sektor_title_dict)

sektor_scope_list = get_scope_filtered_list(df_old, df_new, 'Tahsis Sektör Adı', scope=1000)
df_pra_sektor_change = create_pra_dist_change(df_old, df_new, 'Tahsis Sektör Adı', sorted_sektor_list, sektor_title_dict)
df_pra_sektor_scope_change = create_pra_dist_change(df_old, df_new, 'Tahsis Sektör Adı', sektor_scope_list, sektor_title_dict, '-'+start_term, study_scope=(1000, 'sektörler'))

In [72]:
kurumsal_sube_scope_list = get_scope_filtered_list(df_old, df_new, 'Şube Adı', 'Kurumsal', 'Şube Türü', 5)
ticari_sube_scope_list = get_scope_filtered_list(df_old, df_new, 'Şube Adı', 'Ticari', 'Şube Türü')
karma_sube_scope_list = get_scope_filtered_list(df_old, df_new, 'Şube Adı', 'Karma', 'Şube Türü')

df_pra_kurumsal_sube_change = create_pra_dist_change(df_old, df_new, 'Şube Adı', kurumsal_sube_scope_list, sort=1)
df_pra_ticari_sube_change = create_pra_dist_change_scope(df_old, df_new, 'Şube Adı', ticari_sube_scope_list)
df_pra_karma_sube_change = create_pra_dist_change_scope(df_old, df_new, 'Şube Adı', karma_sube_scope_list)

In [73]:
# quick_export(df_pra_old_max, f'PRA Tüm Firmalar {start_term}', f'Tüm Firmalar {start_term}', False)
# quick_export(df_new, f'PRA Tüm Firmalar {end_term}', f'Tüm Firmalar {end_term}', False)

# Başlıklar

In [76]:
sheet_dict = {
    'Geçişler': df_pra_pass,
    'Modeller': df_pra_model,
    'Segment Geçiş': df_pra_segment_pass,
    'Bölüm Geçiş': df_pra_bolum_pass,
    'Sektör Geçiş': df_pra_sektor_pass,
    'Şube Geçiş': df_pra_sube_pass,
    'Bölge Geçiş': df_pra_bolge_pass,
    'Sektör Dağılım': df_pra_sektor_count_risk_change,
    'Sektör Tüm': df_pra_sektor_change,
    'Sektör': df_pra_sektor_scope_change,
    'Segment': df_pra_segment_change,
    'Bölüm': df_pra_bolum_change,
    'EDS': df_pra_eds_change,
    'Bölge': df_pra_bolge_change,
    'Şube': df_pra_sube_change,
    'Kurumsal': df_pra_kurumsal_sube_change,
    'Ticari': df_pra_ticari_sube_change,
    'Karma': df_pra_karma_sube_change,
}

eds = 'Entegre Derece / Skor'
yzrs = 'Yapay Zeka Risk Sınıfı'
yrpo = 'Yüksek Riskli Portföy Oranı'

merge_dict = {
    'Modeller': {eds: ['D2:E2', 'D13:E13', 'D27:E27', 'D38:E38'], yzrs: ['N2:O2', 'N13:O13', 'N27:O27', 'N38:O38']},
    'Sektör Tüm': {yrpo: ['D3:F3', 'D45:F45']},
    'Sektör': {yrpo: ['D3:F3', 'D28:F28']},
    'Segment': {yrpo: ['D3:F3', 'D16:F16']},
    'Bölüm': {yrpo: ['D3:F3', 'D15:F15']},
    'EDS': {yrpo: ['D3:F3', 'D15:F15']},
    'Bölge': {yrpo: ['D3:F3', 'D26:F26']},
    'Şube': {yrpo: ['D3:F3', 'D15:F15']},
    'Kurumsal': {yrpo: ['D3:F3', 'D23:F23']},
    'Ticari': {yrpo: ['D3:F3', 'D33:F33', 'L3:N3', 'L33:N33']},
    'Karma': {yrpo: ['D3:F3', 'D33:F33', 'L3:N3', 'L33:N33']},
}


sheet_dict_list = [sheet_dict]

sheet_list = [item for sublist in [list(x.keys()) for x in sheet_dict_list] for item in sublist]
df_list = [item for sublist in [list(x.values()) for x in sheet_dict_list] for item in sublist]
# sheet_list = list(range(len(page_list)))

# sheet_list_list = []
# ll = [len(x.keys()) for x in sheet_dict_list]
# c = 0
# for i, l in enumerate(ll):
#     sheet_list_list.append(list(range(c, c + l)))
#     c += l

# pra_sheet_list = sheet_list_list


# df_index = pd.DataFrame()
# sheet_list = [x for x in range(len(page_list))]
# df_index['Sayfa'] = sheet_list
# df_index['Başlık'] = df_index['Sayfa'].apply(lambda x: '=HYPERLINK("#{}!A1", "{}")'.format(str(x), page_list[x]))
# sheet_color_list = ['E5E1E6', 'C7CEEA', 'B2CBF7', 'B5EAD7', 'E2F0CB', 'F7ECC8', 'FFDAC1', 'FFB7B2', 'FF9AA2', 'DCB5CB', 'AF8FC1']

# Final XLSX

In [77]:
output_file_name = f'PRA {pra_v} Val {start_term}-{end_term}'
open_file = True

color_long_multi_trend, color_multi_trend, color_trend, row_based = True, True, True, True
output_file_path = output_folder_path + output_file_name + '.xlsx'
writer = pd.ExcelWriter(output_file_path, engine = 'xlsxwriter')

workbook = writer.book
# bold_format = workbook.add_format({'bold': True, 'text_wrap': True})
# row_format = workbook.add_format({'align': 'left'})
# link_format = workbook.add_format({'underline': True, 'color': 'blue'})
# float_format = workbook.add_format()
# float_format.set_num_format('#,##0.00')
# int_format = workbook.add_format()
# int_format.set_num_format('#,##0')

# color_format_list = []
# for i in range(len(sheet_list_list)):
#     color = sheet_color_list[i]
#     cf = workbook.add_format()
#     cf.set_bg_color(color)
#     color_format_list.append(cf)

# colored_title_format = workbook.add_format({'bold': True})
# colored_title_format.set_bg_color('C9C4CF')


# df_index.to_excel(writer, sheet_name = 'Başlıklar', index = False)
for sheet, df in zip(sheet_list, df_list):
    # sheet = str(i)
    df.to_excel(writer, sheet_name = sheet, index = False)

    worksheet = writer.sheets[sheet]
    worksheet.hide_gridlines(2)

    if sheet in merge_dict:
        for cell_value in merge_dict[sheet]:
            for cell_range in merge_dict[sheet][cell_value]:
                worksheet.merge_range(cell_range, cell_value)
        


    # for j, c in enumerate(df.columns):
    #     worksheet.set_row(0, 60, bold_format)
    #     worksheet.set_column(0, 0, 18, row_format)
    #     if any(x in str(c).lower() for x in ['%', 'ortalama', '2022', '2023', 'n-']):
    #         worksheet.set_column(j, j, 10, float_format)
    #     else:
    #         worksheet.set_column(j, j, 16, int_format)


# worksheet = writer.sheets['Başlıklar']
# worksheet.set_row(0, None, bold_format)
# worksheet.set_column(1, 1, 70, link_format)
# worksheet.write('A1', df_index.columns[0], colored_title_format)
# worksheet.write('B1', df_index.columns[1], colored_title_format)

# for x, cf in enumerate(color_format_list):
#     for i in sheet_list_list[x]:
#         worksheet.write(f'A{i + 2}', i, cf)
#         worksheet.write(f'B{i + 2}', df_index.loc[i, 'Başlık'], cf)

writer.close()

if open_file:
    os.startfile(output_file_path)